# SO-100 Fetch: GPU training with MuJoCo Warp + PyTorch PPO

Physics runs in MuJoCo Warp and PPO in PyTorch, all on the GPU (no per-step CPU round trips).
A Kaggle GPU session has **two T4s** and each training run uses one, so this notebook trains up to **two experiments
side by side** (e.g. two reward functions). Code is cloned from the `warp-training` branch.

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **On**, then **Save Version → Save & Run All** (12 h limit).
Download `results.zip` from the version's Output tab.

**Your own reward:** add a class to `so_arm_warp/rewards.py` on your branch (see the docstring there), push it, set
`BRANCH` below and use its name in `EXPERIMENTS`.

**Physics (`case`):** `full` = all original collision shapes with a cheaper solver (default), `original` = exactly the
CPU env's physics (slower), `filtered` / `simplified` = cheaper collisions (fastest). See `so_arm_warp/physics.py`.

**Send jobs from the command line instead:** `python tools/kaggle_job.py submit ...` (see docs/KAGGLE.md).

**Watch a trained policy locally:** `python -m so_arm_warp.play results/<run>/model_final.pt` (CPU, no GPU needed).

In [ ]:
BRANCH = "warp-training"
HOURS = 11.0                      # stop each run after this long (Kaggle ends sessions at 12 h)
NUM_ENVS = 8192                   # parallel envs per GPU
EXPERIMENTS = [                   # one per GPU; remove one to train a single run
    dict(run_name="carl-default-1", reward="default", case="full"),
    dict(run_name="carl-exact-1", reward="default", case="original"),
]
RESUME = {}                      # e.g. {"carl-default-1": "/kaggle/input/<prev-job>/results/carl-default-1/model_final.pt"}

In [ ]:
%pip install -q "mujoco==4.0.1" "mujoco-warp==4.0.1"

In [ ]:
import os, subprocess, sys, shutil, time, glob
SRC, OUT = "/kaggle/working/src", "/kaggle/working/results"
shutil.rmtree(SRC, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/HumanED/so100.git", SRC], check=True)
print(subprocess.run(["git", "-C", SRC, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
import torch; n_gpu = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(n_gpu)])
assert n_gpu >= 1, "No GPU: set the accelerator to GPU T4 x2"
os.makedirs(OUT, exist_ok=True)

## Train (one process per GPU)

In [ ]:
procs = []
for i, ex in enumerate(EXPERIMENTS[:n_gpu]):
    cmd = [sys.executable, "-m", "so_arm_warp.train", "--run-name", ex["run_name"], "--reward", ex["reward"],
           "--case", ex.get("case", "full"), "--num-envs", str(NUM_ENVS), "--hours", str(HOURS),
           "--out-dir", OUT, "--device", f"cuda:{i}"]
    ck = RESUME.get(ex["run_name"])
    if ck and os.path.exists(ck):
        cmd += ["--resume", ck]
    elif ck:
        print("no checkpoint at", ck, "- starting", ex["run_name"], "from scratch")
    log = open(os.path.join(OUT, f"{ex['run_name']}.stdout"), "w")
    procs.append((ex["run_name"], subprocess.Popen(cmd, cwd=SRC, stdout=log, stderr=subprocess.STDOUT), log))
    print("started", ex["run_name"], "on", f"cuda:{i}")
if len(EXPERIMENTS) > n_gpu:
    print("Only", n_gpu, "GPU(s): skipped", [e["run_name"] for e in EXPERIMENTS[n_gpu:]])

def last_line(name):
    try:
        lines = [l for l in open(os.path.join(OUT, f"{name}.stdout")).read().splitlines() if l.startswith("[")]
        return lines[-1] if lines else "(starting)"
    except FileNotFoundError:
        return "(no output yet)"

while any(p.poll() is None for _, p, _ in procs):
    time.sleep(300)
    for name, p, _ in procs:
        print(f"{name}: {last_line(name)}", flush=True)
for name, p, log in procs:
    log.close()
    print(name, "exit code", p.returncode)
    if p.returncode:
        print(open(os.path.join(OUT, f"{name}.stdout")).read()[-3000:])

## Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
cols = ["ep_return", "grasp_rate", "success_rate", "steps_per_s"]
fig, axes = plt.subplots(1, len(cols), figsize=(4.5 * len(cols), 3.4))
for ex in EXPERIMENTS:
    path = os.path.join(OUT, ex["run_name"], "log.csv")
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    for ax, c in zip(axes, cols):
        if c in df:
            ax.plot(df["step"], df[c], label=ex["run_name"]); ax.set_title(c); ax.set_xlabel("env steps"); ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, "curves.png"), dpi=120); plt.show()

## Evaluate on the exact original physics (CPU MuJoCo) + video

In [ ]:
os.environ.setdefault("MUJOCO_GL", "egl")
for ex in EXPERIMENTS:
    ck = os.path.join(OUT, ex["run_name"], "model_final.pt")
    if not os.path.exists(ck):
        continue
    for extra in (["--physics", "original", "--episodes", "3", "--headless"], ["--episodes", "1", "--video", os.path.join(OUT, f"{ex['run_name']}.mp4")]):
        r = subprocess.run([sys.executable, "-m", "so_arm_warp.play", ck] + extra, cwd=SRC, capture_output=True, text=True,
                           env={**os.environ, "CUDA_VISIBLE_DEVICES": ""})
        print(r.stdout[-1500:] or r.stderr[-800:])

## Package outputs

In [ ]:
shutil.make_archive("/kaggle/working/results", "zip", OUT)
print("Wrote /kaggle/working/results.zip", f"({os.path.getsize('/kaggle/working/results.zip') / 1e6:.1f} MB)")